# Generalizability Evaluation: Function Vectors in Large Language Models

## Overview
This notebook evaluates the generalizability of the findings from the Function Vectors research (Todd et al., ICLR 2024).

### Evaluation Checklist:
- **GT1**: Generalization to a New Model
- **GT2**: Generalization to New Data
- **GT3**: Method/Specificity Generalizability

### Key Findings from Original Research:
- Function vectors are compact representations of input-output functions encoded in attention heads
- They cluster in middle layers (~L/3 depth)
- Models used in original work: GPT-J, GPT-NeoX, Llama 2 (7B, 13B, 70B), GPT-2 XL

In [1]:
# Setup environment and working directory
import os
os.chdir('/home/smallyan/eval_agent')

# Load bashrc for environment variables
bashrc_path = os.path.expanduser('~/.bashrc')
with open(bashrc_path) as f:
    for line in f:
        line = line.strip()
        if line.startswith('export ') and '=' in line:
            var_def = line[7:]  # Remove 'export '
            if '=' in var_def:
                key, value = var_def.split('=', 1)
                value = value.strip('"').strip("'")
                os.environ[key] = value

# Set CC for triton compilation
os.environ['CC'] = '/usr/bin/gcc'
os.environ['TRITON_CACHE_DIR'] = '/tmp/triton_cache'

print("HF_HOME:", os.environ.get('HF_HOME', 'Not set'))
print("TRANSFORMERS_CACHE:", os.environ.get('TRANSFORMERS_CACHE', 'Not set'))

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

HF_HOME: /net/projects2/chai-lab/shared_models
TRANSFORMERS_CACHE: /net/projects2/chai-lab/shared_models/hub


CUDA available: True
GPU: NVIDIA H100 NVL
GPU Memory: 99.95 GB


In [2]:
# Import core libraries
import torch
import json
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from baukit import TraceDict

# Define core utility functions for function vector extraction and intervention
def get_module(model, name):
    """Finds the named module within the given model."""
    for n, m in model.named_modules():
        if n == name:
            return m
    raise LookupError(name)

def add_function_vector(edit_layer, fv_vector, device, idx=-1):
    """Adds a vector to the output of a specified layer in the model."""
    def add_act(output, layer_name):
        current_layer = int(layer_name.split(".")[2])
        if current_layer == edit_layer:
            if isinstance(output, tuple):
                output[0][:, idx] += fv_vector.to(device)
                return output
            else:
                return output
        else:
            return output
    return add_act

def load_pythia_model(model_name, device='cuda'):
    """Load Pythia model with configuration."""
    print(f"Loading: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16).to(device)
    
    MODEL_CONFIG = {
        "n_heads": model.config.num_attention_heads,
        "n_layers": model.config.num_hidden_layers,
        "resid_dim": model.config.hidden_size,
        "name_or_path": model.config.name_or_path,
        "attn_hook_names": [f'gpt_neox.layers.{layer}.attention.dense' for layer in range(model.config.num_hidden_layers)],
        "layer_hook_names": [f'gpt_neox.layers.{layer}' for layer in range(model.config.num_hidden_layers)],
        "prepend_bos": False
    }
    
    return model, tokenizer, MODEL_CONFIG

print("Utility functions defined successfully!")

/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Utility functions defined successfully!


## GT1: Generalization to a New Model

**Original Models Used:** GPT-J (6B), GPT-NeoX (20B), Llama 2 (7B, 13B, 70B), GPT-2 XL

**Test Model:** Pythia-2.8B (EleutherAI) - NOT used in the original paper

We will test if the function vector findings generalize to Pythia-2.8B by:
1. Loading the Pythia model
2. Creating ICL prompts for the antonym task
3. Extracting function vectors from attention heads
4. Testing if function vectors enable zero-shot task execution

In [3]:
# Load the Pythia-2.8B model (NOT used in the original paper)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, tokenizer, model_config = load_pythia_model("EleutherAI/pythia-2.8b", device=device)

print(f"\nModel Configuration:")
print(f"  Layers: {model_config['n_layers']}")
print(f"  Heads: {model_config['n_heads']}")
print(f"  Hidden Dim: {model_config['resid_dim']}")
print(f"  Device: {model.device}")

Loading: EleutherAI/pythia-2.8b


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.68G [00:00<?, ?B/s]

In [4]:
# Check model is loaded and print config
print(f"\nModel Configuration:")
print(f"  Layers: {model_config['n_layers']}")
print(f"  Heads: {model_config['n_heads']}")
print(f"  Hidden Dim: {model_config['resid_dim']}")
print(f"  Device: {model.device}")
print(f"  Optimal edit layer (L/3): {model_config['n_layers']//3}")

In [5]:
# Check output - rerun the print
import sys
print(f"Python version: {sys.version}", flush=True)
print(f"Model type: {type(model)}", flush=True)

In [6]:
print("test")